In [1]:
import sys
from pathlib import Path

# プロジェクトルート
project_root = Path(r"C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project")

# Python がソースコードを探す場所として src/ を追加
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

print("sys.path に追加された src:", src_path)

# ✅ ここがポイント！
# src を sys.path に入れたので、「ifc_ingestor」がトップレベルパッケージになる
# → import するときは「from ifc_ingestor...」で OK（"src." は付けない）
from ifc_ingestor.ingestor import load_ifc_bundle

print("IMPORT SUCCESS:", load_ifc_bundle)

sys.path に追加された src: C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project\src
IMPORT SUCCESS: <function load_ifc_bundle at 0x000001F96F7EDDA0>


In [2]:

# --- IFC の読み込み ---
ifc_path = project_root / "data" / "ifc" / "raw" / "51M-SKA-00-00-M3-MEP-0001_IFC 2x3-P01.ifc"
bundle = load_ifc_bundle(str(ifc_path))

print("\n=== IFC Loaded ===")
print("schema:", bundle.schema)
print("model type:", type(bundle.model))


# --- BDNS タグ抽出 ---
from bdns_extractor.extractor import extract_bdns_tags

bdns_assets = extract_bdns_tags(bundle)

print("\n=== BDNS Extraction Result ===")
print("BDNS-tagged assets:", len(bdns_assets.items))

# 最初の数件を表示
for asset in bdns_assets.items[:10]:
    print(
        "GUID:", asset.ifc_guid,
        "| BDNS:", asset.bdns_tag,
        "| Name:", asset.name,
        "| Class:", asset.raw_ifc_class
    )


=== IFC Loaded ===
schema: IFC2X3
model type: <class 'ifcopenshell.file.file'>

=== BDNS Extraction Result ===
BDNS-tagged assets: 6511
GUID: 2HwPth0ED3muP_ET7DKBaz | BDNS: outlet - double switched socket outlet | Name: DSSO-00001 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK99g | BDNS: outlet - double switched socket outlet | Name: DSSO-00002 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9CM | BDNS: outlet - double switched socket outlet | Name: DSSO-00003 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9Ws | BDNS: outlet - double switched socket outlet | Name: DSSO-00004 | Class: IfcFlowTerminal
GUID: 2HwPth0ED3muP_ET7DK9_L | BDNS: outlet - double switched socket outlet | Name: DSSO-00005 | Class: IfcFlowTerminal
GUID: 1zhYdOoOz9rBBDvGWES7lq | BDNS: outlet - double switched socket outlet | Name: DSSO-00006 | Class: IfcFlowTerminal
GUID: 1zhYdOoOz9rBBDvGWES7qD | BDNS: outlet - double switched socket outlet | Name: DSSO-00007 | Class: IfcFlowTerminal
GUID: 3ID7pTUZ96xeg742M

In [4]:
import json
from dataclasses import asdict, is_dataclass

In [5]:
out_dir = project_root / "data" / "intermediate" / "stage1_ir"
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "bdns_assets.jsonl"

In [6]:

records = []
for item in bdns_assets.items:
    if is_dataclass(item):
        records.append(asdict(item))   # dataclass → dict
    else:
        records.append(dict(item))     # すでに dict の場合

# --- JSON Lines 形式で保存 ---
with out_path.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"written: {out_path}")
print("records:", len(records))


written: C:\Users\Owner\Documents\25952501_RioSaijo\IoT-Interoperability-Project\data\intermediate\stage1_ir\bdns_assets.jsonl
records: 6511
